# Lab: Building a Reproducible Analysis Workflow

In the first part of this lab, you will use official National Park Service data to answer a guided question. In the second part, you will find a dataset and conduct an analysis of your own.

Use the workflow from lecture throughout the lab:

1. **Question**
2. **Data**
3. **Operation**
4. **Check**
5. **Evidence**
6. **Conclusion**
7. **Limitation**


## Learning goals

By the end of this lab, you should be able to:

- organize an investigation with the reproducible analysis workflow;
- inspect a dataset before analyzing it;
- group, summarize, check, and sort observations with Pandas;
- formulate a research question that can be answered with data; and
- distinguish evidence, conclusions, and limitations.


## Using AI tools responsibly

You may use AI to ask questions, debug code, or receive feedback. You remain responsible for understanding your work, verifying suggestions against the data, and citing the source of your dataset. At the end, disclose whether you used an AI tool. Using AI is not required.


# Part 1: Florida park visitation


## Background

The National Park Service (NPS) publishes estimates of recreational visits to its units. These estimates help parks understand patterns in public use and plan staffing, maintenance, transportation, and visitor services.

This dataset contains monthly estimates for five Florida NPS units from 2015 through 2025:

- Big Cypress National Preserve
- Biscayne National Park
- Canaveral National Seashore
- Dry Tortugas National Park
- Everglades National Park

A recreational visit is a visit, not necessarily one unique person. A person who visits several times may be counted several times.

The dataset was prepared from the National Park Service Public Use Statistics Office's [Visitor Use Statistics](https://irma.nps.gov/Stats/Reports/Park), specifically the “Recreation Visitors By Month (1979 - Last Calendar Year)” reports for the five parks. It contains all 660 park-month records from January 2015 through December 2025; no selected park-months were removed, and none of the six retained features are missing. The five park reports were combined and reshaped into one table for the lab.


| Feature | Description |
| --- | --- |
| `park_code` | Four-letter NPS unit code |
| `park` | Name of the NPS unit |
| `year` | Calendar year of the observation |
| `month_number` | Month represented as an integer from 1 through 12 |
| `month` | Month name |
| `recreation_visits` | Estimated recreational visits during that park-month |


## Step 1: Question

**After the decline in July 2020, when did Canaveral National Seashore's July recreational visitation first return to or exceed its average July visitation from 2015–2019?**

The 2015–2019 mean provides a pre-pandemic comparison point. In this question, “recovery” means the first July from 2020 onward with visitation at or above that baseline. This definition does not require every later July to remain above the baseline.


## Step 2: Data

Import Pandas using the alias `pd`. Read `data/nps_florida_park_visitation_2015_2025.csv` into a DataFrame named `park_visits`, then display its first five rows.


In [1]:
# Write your code here.
import pandas as pd

park_visits = pd.read_csv("data/nps_florida_park_visitation_2015_2025.csv")
park_visits.head()

,park_code,park,year,month_number,month,recreation_visits
0,BICY,Big Cypress National Preserve,2015,1,January,114045
1,BICY,Big Cypress National Preserve,2015,2,February,144112
2,BICY,Big Cypress National Preserve,2015,3,March,153084
3,BICY,Big Cypress National Preserve,2015,4,April,113880
4,BICY,Big Cypress National Preserve,2015,5,May,72510


Display the last five rows.


In [2]:
# Write your code here.
park_visits.tail()

,park_code,park,year,month_number,month,recreation_visits
655,EVER,Everglades National Park,2025,8,August,50464
656,EVER,Everglades National Park,2025,9,September,32290
657,EVER,Everglades National Park,2025,10,October,19008
658,EVER,Everglades National Park,2025,11,November,53416
659,EVER,Everglades National Park,2025,12,December,70926


Display information about the DataFrame and calculate the number of missing observations in each feature.


In [3]:
# Write your code here.
park_visits.info()
park_visits.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   park_code          660 non-null    object
 1   park               660 non-null    object
 2   year               660 non-null    int64 
 3   month_number       660 non-null    int64 
 4   month              660 non-null    object
 5   recreation_visits  660 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 31.1+ KB


park_code            0
park                 0
year                 0
month_number         0
month                0
recreation_visits    0
dtype: int64

**What does each row represent?**

Your answer:
Each row represents the recreational visits for one park during a specific month and year.

**Are any observations missing from the six features?**

Your answer: No, there are no missing observations in any of the six features.


## Step 3: Operation

Create one Boolean condition that identifies Canaveral National Seashore and another that identifies July.


In [4]:
# Write your code here.
canaveral_condition = park_visits["park"] == "Canaveral National Seashore"
july_condition = park_visits["month"] == "July"

Combine the two conditions with `&`. Use `.loc` to select the `year` and `recreation_visits` features, make a copy named `canaveral_july`, and display it.


In [5]:
# Write your code here.
canaveral_july = park_visits.loc[
    canaveral_condition & july_condition,
    ["year", "recreation_visits"]
].copy()

canaveral_july

,year,recreation_visits
270,2015,163169
282,2016,216437
294,2017,138018
306,2018,189285
318,2019,162947
330,2020,112898
342,2021,206636
354,2022,233352
366,2023,173296
378,2024,176242


Create `pre_pandemic_condition`, which is `True` for years from 2015 through 2019. Use it with `.loc` to calculate the mean of `recreation_visits`, and store the result as `pre_pandemic_july_mean`.

Display `pre_pandemic_july_mean`.


In [6]:
# Write your code here.
pre_pandemic_condition = canaveral_july["year"].between(2015, 2019)

pre_pandemic_july_mean = canaveral_july.loc[
    pre_pandemic_condition,
    "recreation_visits"
].mean()

pre_pandemic_july_mean

np.float64(173971.2)

Create a condition that identifies rows from 2020 onward whose `recreation_visits` are greater than or equal to `pre_pandemic_july_mean`. Use `.loc` to create `recovery_years`, then display it in ascending year order.


In [7]:
# Write your code here.
recovery_condition = (
    (canaveral_july["year"] >= 2020) &
    (canaveral_july["recreation_visits"] >= pre_pandemic_july_mean)
)

recovery_years = canaveral_july.loc[recovery_condition].sort_values("year")

recovery_years

,year,recreation_visits
342,2021,206636
354,2022,233352
378,2024,176242
390,2025,253403


## Step 4: Check

Display the number of rows in `canaveral_july`, its smallest year, and its greatest year.


In [8]:
# Write your code here.
len(canaveral_july), canaveral_july["year"].min(), canaveral_july["year"].max()

(11, np.int64(2015), np.int64(2025))

**Do these results confirm that the filtered data contains one July observation for every year from 2015 through 2025?**

Your answer: Yes, there are 11 observations covering 2015 through 2025, so there is one July observation for each year.


Count the observations used to calculate `pre_pandemic_july_mean`.


In [9]:
# Write your code here.
canaveral_july.loc[pre_pandemic_condition, "recreation_visits"].count()

np.int64(5)

**Does the baseline use the expected five years?**

Your answer: Yes, the baseline uses five observations, one for each year from 2015 through 2019.


## Step 5: Evidence

Display the first row of `recovery_years`.


In [10]:
# Write your code here.
recovery_years.head(1)

,year,recreation_visits
342,2021,206636


**What was the 2015–2019 mean, and which year first met or exceeded it after the July 2020 decline? Report that year's July visitation.**

Your answer: The 2015–2019 mean was 173,971.2 visits. The first year to meet or exceed it was 2021, with 206,636 visits in July.


## Step 6: Conclusion


**Write one sentence that answers the research question using the evidence.**

Your answer: Canaveral National Seashore's July visitation first recovered above the pre-pandemic average in 2021.


## Step 7: Limitation


**Does first exceeding the baseline mean that every later July remained above it? Use the data to explain.**

Your answer: No. Visitation first exceeded the baseline in 2021, but in 2023 it dropped to 173,296 visits, which was slightly below the 173,971.2 baseline.


**Why does this analysis alone not establish that the pandemic caused the 2020 decline?**

Your answer: This analysis only shows that visitation declined in 2020. It does not prove what caused the decline, since other factors could have affected visitation too.


# Part 2: Your own investigation


## Find a dataset

Find a CSV dataset that interests you. Kaggle is one possible source, but you may use another reputable source. Choose a dataset that:

- has a clearly identified publisher or creator;
- includes documentation explaining the observations and features;
- contains at least one categorical and one quantitative feature;
- is appropriate to share in a class assignment;
- does not contain private or personally identifying information; and
- is small enough to run comfortably in this notebook.

Save the CSV in the notebook's `data` folder before continuing.


**What is the dataset's title, publisher or creator, and direct source URL?**

Your answer: The dataset is "Video Game Sales," created by Gregory Smith and published on Kaggle. Source: https://www.kaggle.com/datasets/gregorut/videogamesales


**What does one row represent?**

Your answer: Each row represents one video game and includes information about its platform, genre, publisher, year, and sales.


## Step 1: Question

Formulate one research question that can be answered with your dataset. The question must require a calculation or comparison, not merely looking up one value.


**What is your research question?**

Your answer: Which video game genre has the highest average global sales?


## Step 2: Data

Read your CSV into a clearly named DataFrame. Display the first and last five rows, information about the DataFrame, and the missing-value count for every feature.


In [11]:
# Write your code here.
video_games = pd.read_csv("data/vgsales.csv")

display(video_games.head())
display(video_games.tail())
video_games.info()
display(video_games.isnull().sum())

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
16593,16596,Woody Woodpecker in Crazy Castle 5,GBA,2002.0,Platform,Kemco,0.01,0.00,0.0,0.0,0.01
16594,16597,Men in Black II: Alien Escape,GC,2003.0,Shooter,Infogrames,0.01,0.00,0.0,0.0,0.01
16595,16598,SCORE International Baja 1000: The Official Game,PS2,2008.0,Racing,Activision,0.00,0.00,0.0,0.0,0.01
16596,16599,Know How 2,DS,2010.0,Puzzle,7G//AMES,0.00,0.01,0.0,0.0,0.01
16597,16600,Spirits & Spells,GBA,2003.0,Platform,Wanadoo,0.01,0.00,0.0,0.0,0.01


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

**Which observations and features will you use to answer your question?**

Your answer:
I will use all 16,598 video games and the Genre and Global_Sales features to compare average global sales by genre.

## Step 3: Operation

Write the code needed to prepare, filter, group, summarize, or sort the data. Store the final evidence in a clearly named object and display it.


In [12]:
# Write your code here.
genre_avg_sales = (
    video_games.groupby("Genre")["Global_Sales"]
    .mean()
    .sort_values(ascending=False)
)

genre_avg_sales

Genre
Platform        0.938341
Shooter         0.791885
Role-Playing    0.623233
Racing          0.586101
Sports          0.567319
Fighting        0.529375
Action          0.528100
Misc            0.465762
Simulation      0.452364
Puzzle          0.420876
Strategy        0.257151
Adventure       0.185879
Name: Global_Sales, dtype: float64

**Explain how your operations connect the selected data to your research question.**

Your answer: I grouped the games by genre, calculated the average global sales for each genre, and sorted the results from highest to lowest.


## Step 4: Check

Write at least one check that could reveal an error in your analysis, then display its result.


In [13]:
# Write your code here.
video_games.groupby("Genre").size().sort_values(ascending=False)

Genre
Action          3316
Sports          2346
Misc            1739
Role-Playing    1488
Shooter         1310
Adventure       1286
Racing          1249
Platform         886
Simulation       867
Fighting         848
Strategy         681
Puzzle           582
dtype: int64

**What did you check, and did the result meet your expectation?**

Your answer: I checked the number of games in each genre to make sure the averages were based on multiple observations. Yes, every genre had hundreds of games.


## Step 5: Evidence


**Which specific value or comparison in your output answers the question?**

Your answer: Platform had the highest average global sales at about 0.94 million units per game.


## Step 6: Conclusion


**What conclusion is supported by the evidence?**

Your answer: Platform games had the highest average global sales among the genres in this dataset


## Step 7: Limitation


**What should a reader avoid concluding from your analysis, and why?**

Your answer: A reader should not conclude that every platform game sells better than games in other genres. This analysis only compares the average sales for each genre.


## AI-use reflection


**State whether you used an AI tool. If you did, describe one suggestion you checked and what you accepted, changed, or rejected. If you did not, state that you did not use one.**

Your response: I used an AI tool to help plan the analysis. I checked its suggestion to compare average global sales by genre and used the dataset output to verify the results before accepting it.


## Submission checklist

Before submitting, confirm that all cells run in order, the independent dataset and its source are included, every answer is complete, numerical claims match the output, conclusions are appropriately limited, and the AI-use reflection is complete.
